installing the packages


In [ ]:
%pip install -r requirements.txt
#% specifies that this command should be exectued in terminal
#-r indicates that a flag we are telling there was a file present install all the packages or requiremtnts that are mentioned in that file

loading the gemini api key

In [2]:
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
load_dotenv()
llm=ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite",
    google_api_key=os.getenv("GEMINI_API_KEY"),
    temperature=0
)


demoOrder db

In [3]:
ORDERS_DB = {
    "12345": {
        "order_id": "12345",
        "customer_email": "john@example.com",
        "items": [
            {
                "name": "Blue Cotton T-Shirt",
                "price": 29.99,
                "quantity": 2
            },
            {
                "name": "Denim Jeans",
                "price": 59.99,
                "quantity": 1
            }
        ],
        "total": 119.97,
        "order_date": "2025-01-15",
        "status": "delivered",
        "tracking_number": "1Z999AA10123456784"
    },

    "67890": {
        "order_id": "67890",
        "customer_email": "jane@example.com",
        "items": [
            {
                "name": "Red Hoodie",
                "price": 45.00,
                "quantity": 1
            }
        ],
        "total": 45.00,
        "order_date": "2025-01-20",
        "status": "shipped",
        "tracking_number": "1Z999AA10123456785"
    },

    "11223": {
        "order_id": "11223",
        "customer_email": "alex@example.com",
        "items": [
            {
                "name": "Wireless Headphones",
                "price": 79.99,
                "quantity": 1
            },
            {
                "name": "USB Cable",
                "price": 9.99,
                "quantity": 2
            }
        ],
        "total": 99.97,
        "order_date": "2025-02-05",
        "status": "processing",
        "tracking_number": "Not yet assigned"
    }
}

tool1:check order status: tools are like function

In [ ]:
from datetime import datetime

def check_order_status(order_id:str)->dict:
    """Looks up and order by its order ID"""
    order=ORDERS_DB.get(order_id)#if order is not presenst it will return none
    if order is None:
        return {
            "error":f"order {order_id} not found",
            "source":"orders_api"
        }
    return {
        "data":order,
        "source":"order_api",
        "retrived_at":datetime.now().isoformat()
    }


{'data': {'order_id': '11223', 'customer_email': 'alex@example.com', 'items': [{'name': 'Wireless Headphones', 'price': 79.99, 'quantity': 1}, {'name': 'USB Cable', 'price': 9.99, 'quantity': 2}], 'total': 99.97, 'order_date': '2025-02-05', 'status': 'processing', 'tracking_number': 'Not yet assigned'}, 'source': 'order_api', 'retrived_at': '2026-11-05T22:01:44.433547'}


tool 2: Refund calculator

In [ ]:
def calculate_refund(order_id:str,reason:str)->dict:
    """calculate potential refund using the demo return policy"""
    order=ORDERS_DB.get(order_id)
    if order is None:
        return {
            "error":"Order not found",
            "source":"refund calculator"
        }
    reason = reason.lower().strip()
    if reason in {"damaged","defective"}:
        fee=0.0
    elif reason=="change_of_mind":
        fee=order["total"]*0.15
    else:
        return {
            "error":"Unsuppoted refund reason",
            "source":"refund_calculate"
        }
    return {
        "data":{
            "order_id":order_id,
            "original_amount":order["total"],
            "restocking_fee":round(fee,2),
            "refund_amount":round(order["total"]-fee,2),
            "reason":reason
        },
        "source":"refund_calculator"
    }


{'data': {'order_id': '11223', 'original_amount': 99.97, 'restocking_fee': 0.0, 'refund_amount': 99.97, 'reason': 'damaged'}, 'source': 'refund_calculator'}


tool 3:user initiatied return -> now we have to process that

In [ ]:
def process_return(order_id:str,reason:str)->dict:
    """check whether a return be initiated checking whether is person eligible or not"""
    order=ORDERS_DB.get(order_id)
    if order is None:
        return {
            "error":"order id is not present or invalid",
            "source":"process_return"
        }
    if order["status"] not in {"delivered"}:
        return {
            "error":"Order has not been delivered",
            "source":"returns_api"
        }
    delivery_date=datetime.strptime(order["delivery_date"],"%Y-%m-%d")
    days_since_delivery=(datetime.now()-delivery_date).days()
    if days_since_delivery>30:
        return {
            "error":"Return window experied (30 days)",
            "source":"procees_return",
            "policy_reference":"return_policy.md.eligibility"
        }
    return {
        "data":{
            "order_id":order_id,
            "return_eligible":True,
            "reason":reason,
            "message":"Eligible for return"
        },
        "source":"returns_api"
    }


{'error': 'Order has not been delivered', 'source': 'returns_api'}


tool4: retrieve a return policy

In [14]:
POLICIES=[{
    "policy":"return_policy",
    "text":("Items can be returned within 30 days of delivery",
            "Damanged or defective items can be returned for a full refund",
            "change of mind returns are subject to a 15% restocking fee."
            )
}
]
def retrieve_policy(query:str)->dict:
    """Retrieves the policy based on the user query"""
    return {
        "data":{
            "query":query,
            "policy":POLICIES
        },
        "source":"knowledge_base"
    }

now contert above functions into langchain tools as they are just functions to convert we have various ways
1.use @tool decorator above each function or below is one more way

In [19]:
from langchain_core.tools import StructuredTool
tools=[
    StructuredTool.from_function(check_order_status),
    StructuredTool.from_function(retrieve_policy),
    StructuredTool.from_function(calculate_refund),
    StructuredTool.from_function(process_return)
]
for tool in tools:
    print(f"Tool: {tool.name}")
    print(f"Description: {tool.description}")
    print()

Tool: check_order_status
Description: Looks up and order by its order ID

Tool: retrieve_policy
Description: Retrieves the policy based on the user query

Tool: calculate_refund
Description: calculate potential refund using the demo return policy

Tool: process_return
Description: check whether a return be initiated



creating a agent state